# Self Attention

## Data Preparation

1) padding <pad> for making all sequences of same length in batch.  
2) unknown <unk> to handle any rare or out of vocabulary words the model might encounter during inference

In [7]:
import math
import os
import re 
import urllib.request
from collections import Counter
from typing import Callable, Dict, List, Optional, Sequence

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn   
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


/home/mfaizan/personal/FROM_SCRATCH_ML/from_scratch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:

# Tiny toy dataset
sentences = """
the dog chased the cat
the cat chased the mouse
the dog ran fast
the mouse ran fast
the cat lay down
"""

## build vocab
tokens = sentences.split()
vocab = ['<pad>','<unk>'] + sorted(set(tokens))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for i, w in enumerate(vocab)}
print(f"Vocab size: {len(vocab)}")

Vocab size: 11


In [9]:
## Tokenizer
## convert into root letters or basic unit from the word 
## and mapping tokens to indices

## Steps
#1) Coverting everything to lower case consistency
#2) Removing punctuation to focus on word embeddings
#3) Splitting on Whitespace and word boundaries


## Tokenizer

In [10]:
class SimpleTokenizer:
    """
    Splits on whitespace and lowercases, with optional regex for real word tokens.
    """

    def __init__(self):
        pass

    def __call__(self, text):
        text = text.lower()
        tokens = re.findall(r'\b\w+\b', text)
        return tokens




In [11]:
tokenizer  = SimpleTokenizer()
tokens = tokenizer("The Dog chased the Cat.")
print(tokens)

['the', 'dog', 'chased', 'the', 'cat']


## Building the Vocabulary

In [13]:
def build_vocab(sentences, tokenizer, min_freq = 1):

    counter = Counter()

    ## dictionary for word count
    for sent in sentences:
        counter.update(tokenizer(sent))

    ## creating vocab
    vocab = [w for w,c in counter.items() if c>=min_freq]
    vocab+=['<pad>','<unk>']

    ## word to index
    word2idx = {w: i for i, w in enumerate(vocab)}

    ## index to word
    idx2word = {i:w for i, w in enumerate(vocab)}

    return vocab, word2idx, idx2word


In [15]:
# Using our sample sentences and tokenizer
sentences = [
    "the dog chased the cat",
    "the cat chased the mouse",
    "the dog ran fast",
    "the mouse ran fast",
    "the cat lay down"
]
tokenizer = SimpleTokenizer()
vocab, word2idx, idx2word = build_vocab(sentences, tokenizer, min_freq=1)
print(f"Vocab: {vocab}")
print(f"Word to Index: {word2idx}")
print(f"Index to Word: {idx2word}")

Vocab: ['the', 'dog', 'chased', 'cat', 'mouse', 'ran', 'fast', 'lay', 'down', '<pad>', '<unk>']
Word to Index: {'the': 0, 'dog': 1, 'chased': 2, 'cat': 3, 'mouse': 4, 'ran': 5, 'fast': 6, 'lay': 7, 'down': 8, '<pad>': 9, '<unk>': 10}
Index to Word: {0: 'the', 1: 'dog', 2: 'chased', 3: 'cat', 4: 'mouse', 5: 'ran', 6: 'fast', 7: 'lay', 8: 'down', 9: '<pad>', 10: '<unk>'}


In [16]:
word = "cat"
idx = word2idx.get(word, word2idx['<unk>'])
print(f"Index of '{word}': {idx}")
print(f"Word for index {idx}: {idx2word[idx]}")

Index of 'cat': 3
Word for index 3: cat


## sliding Windows

**Predicting Next Word after Window**

In [19]:
## selecting fixed size window as input and train to predict the next word that follows the window in the sentence.
tokenizer = SimpleTokenizer()
vocab, word2idx, idx2word = build_vocab(sentences, tokenizer)

## 2
SEQ_LEN = 4

## 3
encoded_sentences = []
for sent in sentences:
    ## tokenizations
    tokens = tokenizer(sent)
    
    ## Map tokens to IDS
    ids = [word2idx.get(tok, word2idx['<unk>']) for tok in tokens]

    encoded_sentences.append(ids)

## 4 
inputs = []
targets = []
for ids in encoded_sentences:
    for i in range(len(ids)-SEQ_LEN):
        ## window length of SEQ_LEN
        window = ids[i:i+SEQ_LEN]

        ## Just next word after target
        target = ids[i+SEQ_LEN]

        inputs.append(window)
        targets.append(target)

## 5
for input , target in zip(inputs, targets):
    inp_words = [idx2word[i] for i in input]
    tgt_word = [idx2word[target]]

    print(f"Input: {inp_words} -> Target : {tgt_word}")



Input: ['the', 'dog', 'chased', 'the'] -> Target : ['cat']
Input: ['the', 'cat', 'chased', 'the'] -> Target : ['mouse']


## Turning the Data into a Pytorch Dataset 

class TinyDataset(Dataset):
